# Módulo 8 · El lakehouse en acción y cierre

⏱ **10 min** · Slides: 3, 6–8 (una sola copia, tres perfiles, gobierno) y 31 (quiz)

**Objetivo:** cerrar el círculo. Los slides prometen *"una sola copia de los datos, abierta y gobernada, para todas las cargas"*. Lo comprobamos.

| Parte | Qué haces |
|---|---|
| 1 | Tres perfiles leen la misma tabla |
| 2 | Gobierno con Unity Catalog |
| 3 | Historial de la tabla |
| 4 | Quiz |
| 5 | Mapa de lo aprendido y siguientes pasos |

**Requiere:** módulo 7.

In [ ]:
%run ./00_Setup

# Parte 1 · Una plataforma, tres perfiles, una sola tabla

### 🛠 Data Engineer (PySpark)
"¿Cuántos episodios llegaron por tipo de red?"

In [ ]:
display(spark.table(SILVER).groupBy("network_type").count().orderBy(F.col("count").desc()))

### 📊 Analista de datos (SQL)
"¿Cuáles son las 10 redes con más episodios?"

In [ ]:
%sql
SELECT network_name, network_type, episodios, series, rating_promedio
FROM gold_resumen_red
ORDER BY episodios DESC
LIMIT 10

### 🔬 Data Scientist (PySpark)
"¿La duración de un episodio se relaciona con su calificación?"

In [ ]:
datos = spark.table(SILVER).select("runtime_min", "episode_rating").dropna()

print("Episodios con duración y calificación:", datos.count())
print("Correlación duración–calificación    :", round(datos.stat.corr("runtime_min", "episode_rating"), 3))
display(datos.summary("count", "mean", "min", "50%", "max"))

👀 **Qué observar:** los tres perfiles usaron datos que viven **en un solo lugar** (`silver_episodios` / `gold_*`).
Nadie copió nada a otro sistema ni mantiene "su versión" de la verdad.

# Parte 2 · Gobierno con Unity Catalog

In [ ]:
# Inventario de todo lo que construimos, desde el catálogo del sistema
display(spark.sql(f"""
    SELECT table_name, table_type, data_source_format
    FROM `{CATALOG}`.information_schema.tables
    WHERE table_schema = '{SCHEMA}'
    ORDER BY table_name
"""))

In [ ]:
# Metadatos de una tabla: propietario, formato Delta, tipo MANAGED, ubicación
display(spark.sql(f"DESCRIBE EXTENDED {GOLD_RED}"))

🧪 **Pruébalo en la interfaz:**
1. **Catalog** → `workspace` → `lab_tvmaze` → `gold_resumen_red`
2. Pestañas **Details**, **Sample Data** y **Lineage** (si está disponible: verás el recorrido Bronze → Silver → Gold)
3. Pestaña **Permissions**: aquí se darían o quitarían accesos a otros usuarios, en un solo lugar para todas las cargas

# Parte 3 · Historial de una tabla Delta

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {SILVER}").select("version", "timestamp", "operation", "operationMetrics"))

👀 **Qué observar:** cada construcción de Silver dejó una versión. Delta guarda quién hizo qué y cuándo.

# Parte 4 · Mini quiz (slide 31)

Responde en voz alta o por escrito. Las respuestas están más abajo.

1. **Lakehouse:** ¿qué ventaja da frente a un data lake solo?
2. **Clusters:** ¿cuándo conviene un job cluster?
3. **DataFrames:** ¿por qué `display()` ejecuta algo y `filter()` no?
4. **Delta:** ¿qué agrega Delta a Parquet?
5. **Ingesta:** ¿cómo evitas duplicados si ejecutas la carga dos veces?

*(Antes de seguir, piensa tu respuesta.)*

### Respuestas

In [ ]:
respuestas = {
    "1. Lakehouse": "Agrega transacciones ACID, calidad y gobierno sobre el almacenamiento barato y abierto del data lake (Módulo 4, pasos 6–8).",
    "2. Job cluster": "En producción y cargas automatizadas: se crea al lanzar el job y se elimina al terminar, así cuesta menos.",
    "3. display vs filter": "filter() es una transformación lazy: solo describe el plan. display() es una acción: dispara la ejecución (Módulo 3, pasos 4–5).",
    "4. Delta": "Un log de transacciones (_delta_log) que da ACID, versiones y time travel (Módulo 4, pasos 6–8).",
    "5. Duplicados": "Cargas idempotentes: registrar los archivos ya cargados o usar MERGE por clave (Módulo 6, parte C).",
}
for pregunta, respuesta in respuestas.items():
    print(f"{pregunta}\n   → {respuesta}\n")

# Parte 5 · Mapa de lo aprendido

| Tema de los slides | Módulo del laboratorio | Evidencia |
|---|---|---|
| Lakehouse, una sola copia, perfiles | 8 | Tres perfiles, una tabla |
| Azure Databricks, workspace, Catalog | 1, 8 | Navegación y Unity Catalog |
| Clusters (driver, workers, particiones) | 1, 3 | `spark_partition_id()` |
| Notebooks (magic, widgets, secretos, `%run`) | 1, 2 | Celdas `%sql`, widgets, `%run ./00_Setup` |
| DataFrames: distribuido, inmutable, lazy | 3 | Tiempos, `explain()`, `Exchange` |
| Esquema inferido vs explícito | 3 | `007 → 7` |
| CSV vs Parquet vs Delta | 4 | Tamaños y `ReadSchema` |
| Delta: log, ACID, time travel | 4 | `_delta_log`, escritura fallida, `RESTORE` |
| Ingesta de múltiples archivos | 5 | Comodín + `_metadata.file_path` + append |
| Tres problemas de ingesta | 6 | Modos de lectura, `mergeSchema`, `MERGE` |
| Bronze → Silver → Gold | 5, 7 | Tablas `bronze_`, `silver_`, `gold_` |

## Para seguir practicando

**1. Automatiza (equivale a un *job cluster* del slide 13).** En el menú lateral abre **Jobs & Pipelines → Create → Job**,
agrega tareas en orden `02_Ingesta_RAW` → `05_Bronze_multiples_archivos` → `07_Silver_y_Gold` y ejecútalo.
En la Free Edition corre sobre cómputo serverless.

**2. Reto de modelado.** Reconstruye con PySpark las dimensiones y el hecho de la práctica `01_ELT`
(`dim_shows`, `dim_networks`, `dim_time`, `fact_episodes`) a partir de `silver_episodios`.

**3. Más datos.** Cambia el widget `dias` del módulo 2 a `30` y vuelve a correr 5 → 7. ¿Cambian las particiones? ¿Y el tiempo?

**Siguiente en el curso:** Módulo 3 · Fundamentos de Delta Lake.

## 🧹 Limpieza (opcional)
Elimina **todo** lo creado por el laboratorio: tablas y archivos del volumen. Quita el `#` para ejecutarla.

In [ ]:
# spark.sql(f"DROP SCHEMA `{CATALOG}`.{SCHEMA} CASCADE")